# A3 03 — Long-term dominance and diversity through time

## Research questions
3. How do long-term dominance patterns compare between Eastern and Western sectors in living and bone assemblages?
4. How do richness, diversity, and evenness vary through time?

The main comparisons use the matched wild-mammal taxon set. A separate complete-bone summary retains unresolved and other mammal categories but is not treated as directly comparable with the aerial survey.


In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'scripts'))
from a3_common import *
processed_dir, output_dir = ensure_a3_directories(ROOT)
aerial = pd.read_pickle(processed_dir / 'aerial_primary.pkl')
bones = pd.read_pickle(processed_dir / 'bones_primary_excluding_2024.pkl')
decisions = pd.read_csv(processed_dir / 'taxon_decisions.csv')
shared_taxa = decisions.loc[decisions['matched_analysis'], 'taxon'].tolist()
aerial_matched = annual_matrix(aerial, shared_taxa)
bones_matched = annual_matrix(bones, shared_taxa)
bones_complete = annual_matrix(bones[~bones['Species'].isin(DOMESTIC_TAXA | BIRD_TAXA)])


## Analysis 1 — Long-term dominance by sector

Dominance is summarized with three transparent quantities: mean annual relative abundance, median annual rank when present, and occupancy (the proportion of sampled years in which a taxon occurs). These replace the custom Borda score.

**Plain-language example.** A taxon can dominate because it is consistently common every year, or because it is extremely common in only one year. Mean abundance, median rank, and occupancy distinguish those patterns.


In [2]:
living_dominance = dominance_summary(aerial_matched).assign(dataset='living')
bone_dominance = dominance_summary(bones_matched).assign(dataset='bones')
dominance = pd.concat([living_dominance, bone_dominance], ignore_index=True)
dominance.to_csv(output_dir / 'A3_03_matched_taxon_dominance.csv', index=False)
top_dominance = dominance.groupby(['dataset','sector'], group_keys=False).head(8)
top_dominance.round(3)


,sector,taxon,mean_relative_abundance,median_annual_rank_when_present,occupancy,years,dataset
0,Eastern,Equus burchellii,0.238,2.00,1.000,20,living
1,Eastern,Aepyceros melampus,0.229,2.00,1.000,20,living
2,Eastern,Syncerus caffer,0.214,2.00,1.000,20,living
3,Eastern,Eudorcas thomsonii,0.087,4.00,1.000,20,living
4,Eastern,Nanger granti,0.072,5.00,1.000,20,living
5,Eastern,Phacochoerus africanus,0.043,6.00,0.950,20,living
6,Eastern,Kobus ellipsiprymnus,0.029,7.00,1.000,20,living
7,Eastern,Giraffa camelopardalis,0.029,9.00,0.900,20,living
15,Western,Equus burchellii,0.365,1.00,1.000,17,living
16,Western,Aepyceros melampus,0.211,2.00,1.000,17,living


### Interpretation and copy-ready table

With the historical Sweetwaters aerial totals included, Eastern living annual composition was shared mainly by plains zebra (mean 23.8%), impala (22.9%), and buffalo (21.4%), all present in every sampled year. Western living assemblages were dominated by plains zebra (36.5%), followed by impala (21.1%) and buffalo (15.5%). Bone MNI was more strongly zebra-dominated: 43.4% in the East and 45.0% in the West. Warthog was the second-ranked bone taxon in both sectors (13.3% East; 14.9% West), followed by Thomson's gazelle and giraffe. Bone dominance therefore only partly reproduces living dominance.


## Analysis 2 — Comparable annual diversity

Richness is the number of represented taxa. Hill N1 is the effective number of common taxa derived from Shannon diversity. Hill N2 emphasizes the most abundant taxa. Hill evenness divides N1 by richness; it approaches 1 when represented taxa are similarly abundant.

**Plain-language example.** A sample containing four equally common taxa has a higher effective diversity than a sample with four taxa in which one accounts for almost every observation.

Bone diversity is interpreted only when summed MNI across eligible matched taxa and contributing, spatially separated transects is at least 10 within one sector × estimated-death-year sample. This does not require 10 taxa and does not refer to skeletal elements, transects, or aerial-census animals. Smaller MNI samples remain visible but are explicitly flagged. No temporal significance test is applied to the sparse bone series.


In [3]:
living_diversity = annual_diversity(aerial_matched, minimum_count=1).assign(dataset='living')
bone_diversity = annual_diversity(bones_matched, minimum_count=10).assign(dataset='bones')
matched_diversity = pd.concat([living_diversity, bone_diversity], ignore_index=True)
matched_diversity.to_csv(output_dir / 'A3_03_matched_annual_diversity.csv', index=False)

summary = (matched_diversity[matched_diversity['interpret']]
           .groupby(['dataset','sector'])
           .agg(years_interpreted=('year','size'), median_count=('total_count','median'),
                richness_min=('richness_N0','min'), richness_max=('richness_N0','max'),
                hill_N1_min=('hill_N1','min'), hill_N1_max=('hill_N1','max'),
                evenness_min=('hill_evenness_N1_over_N0','min'), evenness_max=('hill_evenness_N1_over_N0','max'))
           .reset_index())
summary.to_csv(output_dir / 'A3_03_diversity_summary.csv', index=False)
summary.round(3)


,dataset,sector,years_interpreted,median_count,richness_min,richness_max,hill_N1_min,hill_N1_max,evenness_min,evenness_max
0,bones,Eastern,11,23.0,3,11,2.401,6.918,0.537,0.939
1,bones,Western,6,33.5,6,10,3.179,6.940,0.477,0.694
2,living,Eastern,20,2047.5,10,15,5.188,7.893,0.412,0.616
3,living,Western,17,6507.0,11,14,3.758,6.755,0.342,0.550


### Interpretation

All 20 Eastern and 17 Western living years were interpretable. Eastern living richness ranged from 10 to 15 taxa and Hill N1 from 5.188 to 7.893; Western richness ranged from 11 to 14 and Hill N1 from 3.758 to 6.755. The lower Eastern richness minimum occurs in the historical whole-Sweetwaters totals, which lack block detail. At the prespecified minimum summed matched-taxon MNI of 10 within one sector × estimated-death-year sample, 11 of 14 Eastern years and 6 of 11 Western years were retained. Their observed Hill N1 ranges were 2.401–6.918 and 3.179–6.940, respectively. Western bone temporal coverage remains limited. These ranges describe variation rather than a tested trend.


## Analysis 3 — Complete bone-only diversity

This supplementary description includes unresolved categories and mammals not represented consistently in the aerial survey. It answers what was recovered in the bone assemblage, not whether bones match the aerial community.


In [4]:
complete_bone_diversity = annual_diversity(bones_complete, minimum_count=10)
complete_bone_diversity.to_csv(output_dir / 'A3_03_complete_bone_diversity.csv', index=False)
complete_bone_summary = (complete_bone_diversity[complete_bone_diversity['interpret']]
    .groupby('sector').agg(years_interpreted=('year','size'), richness_min=('richness_N0','min'),
                           richness_max=('richness_N0','max'), hill_N1_min=('hill_N1','min'),
                           hill_N1_max=('hill_N1','max')).reset_index())
complete_bone_summary.round(3)


,sector,years_interpreted,richness_min,richness_max,hill_N1_min,hill_N1_max
0,Eastern,12,5,15,3.804,9.638
1,Western,7,5,16,4.353,9.070


## Overall interpretation

**Copy-ready result.** With three historical Sweetwaters aerial totals added, Eastern living assemblages were led by plains zebra, impala, and buffalo, whereas plains zebra remained the leading Western taxon. Bone MNI was zebra-dominated in both sectors (mean annual relative abundance: East 43.4%, West 45.0%), with warthog second (13.3% and 14.9%). Living Hill N1 diversity ranged from 5.188 to 7.893 across 20 Eastern years and from 3.758 to 6.755 across 17 Western years. Requiring summed matched-taxon MNI of at least 10 within each sector × estimated-death-year sample, bone Hill N1 ranged from 2.401 to 6.918 across 11 Eastern years and from 3.179 to 6.940 across 6 Western years. Complete wild-mammal bone-only summaries, which exclude cattle and birds but retain unresolved categories, yielded Hill N1 ranges of 3.804–9.638 across 12 Eastern years and 4.353–9.070 across 7 Western years. Temporal ranges remain descriptive because the weathering-derived years are unevenly distributed and Western replication is limited.
